# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# Kather-to-CRC common-seven attribution faithfulness

This notebook can run concurrently with notebook 12 on a two-GPU machine.
Notebook 12 uses GPU 0; this notebook selects GPU 1 when available. It waits for a
model-specific completion marker before reading predictions or checkpoints, then
starts attribution for that model while notebook 12 trains the next model.

The evaluation is intentionally focused:

- the prediction-independent 98-image cohort frozen by notebook 11;
- ResNet18 Grad-CAM, DINOv2 gradient-weighted rollout, and UNI
  gradient-weighted rollout;
- common 14 x 14 image-space evaluation;
- normalized-zero replacement;
- five random deletion repeats;
- common true-class targets for all cohort images;
- the jointly correct subset as the primary paired comparison.

This is a model-prediction intervention study. Highlighted patches are regions
contributing to predictions, not biological causes.


In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display


def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PREFLIGHT_DIR = PROJECT_ROOT / 'artifacts' / 'crc_val_external' / 'preflight'
COMMON_DIR = PROJECT_ROOT / 'artifacts' / 'crc_val_external' / 'common_seven'
CLASSIFICATION_DIR = COMMON_DIR / 'classification'
CHECKPOINT_DIR = COMMON_DIR / 'checkpoints'
OUTPUT_DIR = COMMON_DIR / 'faithfulness'
STATISTICS_DIR = COMMON_DIR / 'statistics'
for directory in (OUTPUT_DIR, STATISTICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('Notebook 13 requires a CUDA GPU')
DEVICE = torch.device('cuda:0')

SEEDS = (11, 89, 181)
MODEL_ORDER = ('ResNet18', 'DINOv2', 'UNI')
WAIT_FOR_CLASSIFICATION = True
MAX_WAIT_HOURS = 12
POLL_SECONDS = 60
OVERWRITE_PARTITIONS = False
INCLUDE_ERROR_PREDICTED_TARGETS = True
BOOTSTRAP_ITERATIONS = 5000

print('Project:', PROJECT_ROOT)
print('Attribution device:', DEVICE)
print('Outputs:', OUTPUT_DIR)


## Frozen cohort and concurrency guard

The cohort filename records whether official patient metadata was available. If
not, inference remains tile-level exploratory and the notebook never treats image
tiles as independent patients.


In [ ]:
from Methods.CNNBenchmark import build_resnet18
from Methods.CRCExternalValidation import (
    ExternalCNNAdapter,
    ExternalTransformerAdapter,
    build_external_target_index,
    finalize_analysis_families,
    paired_external_inference,
    recompute_live_cohort_predictions,
)
from Methods.DINOv2Attribution import (
    build_dinov2_classifier,
    resolve_dinov2_transform,
)
from Methods.KatherRevision.evaluation import (
    consolidate_partitions,
    evaluate_image_seed_partitions,
)
from Methods.UNIAttribution import build_uni_classifier, resolve_uni_transform

readiness = json.loads(
    (PREFLIGHT_DIR / 'preflight_readiness.json').read_text(encoding='utf-8')
)
cohort_suffix = (
    'patient_stratified'
    if readiness['patient_metadata_available']
    else 'tile_exploratory'
)
cohort_path = (
    PREFLIGHT_DIR / f'common_seven_attribution_cohort_{cohort_suffix}.csv'
)
if not cohort_path.is_file():
    legacy_path = PREFLIGHT_DIR / 'common_seven_attribution_cohort.csv'
    if legacy_path.is_file():
        cohort_path = legacy_path
    else:
        raise FileNotFoundError(cohort_path)
cohort = pd.read_csv(cohort_path)
assert len(cohort) == 98
assert cohort['common_class'].nunique() == 7
assert not cohort['selected_using_predictions'].astype(bool).any()
assert not cohort['selected_using_attributions'].astype(bool).any()
print('Cohort:', cohort_path)
print('Inference level:', readiness['inference_level'])

MODEL_FILES = {
    'ResNet18': (
        CLASSIFICATION_DIR / 'resnet18.complete',
        CLASSIFICATION_DIR / 'resnet18_external_predictions.csv',
        CHECKPOINT_DIR / 'resnet18',
    ),
    'DINOv2': (
        CLASSIFICATION_DIR / 'dinov2.complete',
        CLASSIFICATION_DIR / 'dinov2_external_predictions.csv',
        CHECKPOINT_DIR / 'dinov2',
    ),
    'UNI': (
        CLASSIFICATION_DIR / 'uni.complete',
        CLASSIFICATION_DIR / 'uni_external_predictions.csv',
        CHECKPOINT_DIR / 'uni',
    ),
}


def model_ready(model_name):
    marker, prediction_path, checkpoint_dir = MODEL_FILES[model_name]
    checkpoints = [checkpoint_dir / f'seed_{seed}.pt' for seed in SEEDS]
    run_complete = (
        marker.is_file()
        or (COMMON_DIR / 'classification_run_metadata.json').is_file()
    )
    return (
        run_complete
        and prediction_path.is_file()
        and prediction_path.stat().st_size > 0
        and all(path.is_file() and path.stat().st_size > 0 for path in checkpoints)
    )


def wait_for_model(model_name):
    if model_ready(model_name):
        return
    if not WAIT_FOR_CLASSIFICATION:
        raise FileNotFoundError(f'{model_name} classification artifacts are incomplete')
    deadline = time.time() + MAX_WAIT_HOURS * 3600
    print(f'Waiting for notebook 12 to complete {model_name}...')
    while time.time() < deadline:
        if model_ready(model_name):
            print(f'{model_name} classification artifacts are ready')
            return
        time.sleep(POLL_SECONDS)
    raise TimeoutError(f'Timed out waiting for {model_name} after {MAX_WAIT_HOURS} hours')


## Focused model-by-model evaluation

Each image x seed partition is saved independently. Rerunning this cell skips
complete partitions. A completion marker is written only after all three seeds of
one model have finished.


In [ ]:
METHODS = {
    'ResNet18': ('gradcam',),
    'DINOv2': ('gradient_attention_rollout',),
    'UNI': ('gradient_attention_rollout',),
}


def make_adapter(model_name):
    checkpoint_dir = MODEL_FILES[model_name][2]
    if model_name == 'ResNet18':
        return ExternalCNNAdapter(
            'ResNet18',
            model_builder=lambda: build_resnet18(
                num_classes=7, pretrained=False, freeze_backbone=False
            ),
            checkpoint_dir=checkpoint_dir,
            device=DEVICE,
            checkpoint_type='state_dict',
            image_size=150,
            native_grid_size=5,
        )
    if model_name == 'DINOv2':
        model = build_dinov2_classifier(7, DEVICE)
        transform, _ = resolve_dinov2_transform(model.encoder)
        return ExternalTransformerAdapter(
            'DINOv2', model, checkpoint_dir, DEVICE, transform, native_grid_size=16
        )
    model = build_uni_classifier(
        PROJECT_ROOT,
        7,
        DEVICE,
        assets_dir=os.environ.get('UNI_ASSETS_DIR') or None,
    )
    transform, _ = resolve_uni_transform(model.encoder)
    return ExternalTransformerAdapter(
        'UNI', model, checkpoint_dir, DEVICE, transform, native_grid_size=14
    )


model_prediction_frames = []
for model_name in MODEL_ORDER:
    wait_for_model(model_name)
    prediction_path = MODEL_FILES[model_name][1]
    model_predictions = pd.read_csv(prediction_path)
    assert set(model_predictions['seed']) == set(SEEDS)
    assert model_predictions.groupby('seed')['relative_path'].nunique().eq(6588).all()
    adapter = make_adapter(model_name)
    live_cohort_predictions = recompute_live_cohort_predictions(
        adapter,
        cohort,
        model_predictions,
        class_names=(
            'adipose', 'background', 'debris_mucus', 'lymphocytes',
            'normal_mucosa', 'stroma', 'tumor',
        ),
    )
    mismatch_count = int(
        live_cohort_predictions['prediction_changed_from_classification'].sum()
    )
    print(
        f'{model_name}: {mismatch_count} / {len(live_cohort_predictions)} '
        'cohort predictions changed in the live attribution forward'
    )
    live_cohort_predictions.to_csv(
        OUTPUT_DIR / f'{model_name.lower()}_live_cohort_predictions.csv', index=False
    )
    model_prediction_frames.append(live_cohort_predictions)

    target_index = build_external_target_index(
        cohort,
        live_cohort_predictions,
        model_name,
        include_error_predicted_targets=INCLUDE_ERROR_PREDICTED_TARGETS,
    )
    target_index.to_csv(
        OUTPUT_DIR / f'{model_name.lower()}_target_index.csv', index=False
    )
    started = time.time()
    evaluate_image_seed_partitions(
        adapter,
        target_index,
        output_dir=OUTPUT_DIR,
        perturbations=('normalized_zero',),
        common_grid_sizes=(14,),
        integrated_gradient_steps=1,
        random_repeats=5,
        random_null_repeats=0,
        batch_size=32,
        include_higher_res_cam=False,
        requested_methods=METHODS[model_name],
        include_native_grid=False,
        overwrite=OVERWRITE_PARTITIONS,
    )
    (OUTPUT_DIR / f'{model_name.lower()}.complete').write_text(
        'all external faithfulness partitions complete\n', encoding='utf-8'
    )
    adapter.model = None
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
    print(f'{model_name} attribution hours: {(time.time() - started) / 3600:.2f}')


## Consolidation and primary analysis family

All-image true-class results are retained as sensitivity results. The primary
family duplicates only images correctly classified by all three models for the
same seed, preserving a common true-class target.


In [ ]:
all_predictions = pd.concat(model_prediction_frames, ignore_index=True)
all_predictions.to_csv(
    CLASSIFICATION_DIR / 'attribution_models_external_predictions.csv', index=False
)

metrics = consolidate_partitions(
    OUTPUT_DIR, 'metrics', OUTPUT_DIR / 'external_faithfulness_metrics_raw.csv'
)
curves = consolidate_partitions(
    OUTPUT_DIR, 'deletion_curves', OUTPUT_DIR / 'external_deletion_curves_raw.csv'
)
maps = consolidate_partitions(
    OUTPUT_DIR, 'attribution_maps', OUTPUT_DIR / 'external_attribution_maps_raw.csv'
)
occlusion = consolidate_partitions(
    OUTPUT_DIR, 'occlusion_scores', OUTPUT_DIR / 'external_occlusion_scores_raw.csv'
)

metrics = finalize_analysis_families(metrics, all_predictions)
curves = finalize_analysis_families(curves, all_predictions)
maps = finalize_analysis_families(maps, all_predictions)
occlusion = finalize_analysis_families(occlusion, all_predictions)
metrics.to_csv(OUTPUT_DIR / 'external_faithfulness_metrics.csv', index=False)
curves.to_csv(OUTPUT_DIR / 'external_deletion_curves.csv', index=False)
maps.to_csv(OUTPUT_DIR / 'external_attribution_maps.csv', index=False)
occlusion.to_csv(OUTPUT_DIR / 'external_occlusion_scores.csv', index=False)

primary = metrics[
    metrics['analysis_family'].eq('primary_jointly_correct_true_class')
    & metrics['perturbation'].eq('normalized_zero')
    & metrics['grid_label'].eq('common_14')
].copy()
primary_methods = {
    'ResNet18': 'gradcam',
    'DINOv2': 'gradient_attention_rollout',
    'UNI': 'gradient_attention_rollout',
}
primary = primary[primary['method'].eq(primary['model'].map(primary_methods))]
assert primary['model'].nunique() == 3
print('Jointly correct image-seed pairs:', primary[['cohort_id', 'seed']].drop_duplicates().shape[0])

all_image = metrics[
    metrics['analysis_family'].eq('all_image_true_class_sensitivity')
    & metrics['perturbation'].eq('normalized_zero')
    & metrics['grid_label'].eq('common_14')
].copy()
all_image = all_image[
    all_image['method'].eq(all_image['model'].map(primary_methods))
]
family_counts = pd.concat(
    (
        primary.assign(reporting_family='jointly_correct_true_class'),
        all_image.assign(reporting_family='all_image_true_class'),
    ),
    ignore_index=True,
).groupby(['reporting_family', 'model', 'class_name'], as_index=False).agg(
    image_seed_rows=('cohort_id', 'size'),
    unique_images=('cohort_id', 'nunique'),
)
family_counts.to_csv(
    STATISTICS_DIR / 'external_analysis_family_class_counts.csv', index=False
)
joint_classes = set(primary['class_name'].unique())
missing_joint_classes = sorted(set(cohort['common_class']) - joint_classes)
print('Classes absent from jointly correct subset:', missing_joint_classes)


## Focused faithfulness statistics

Seeds are averaged within images before paired model comparisons. When official
patient IDs are present, the bootstrap resamples patients first and images second.
Without them, intervals are explicitly labeled paired tile-level exploratory
bootstrap intervals. Wilcoxon tests are paired and Holm-corrected within metric.


In [ ]:
primary_metrics = [
    'attribution_occlusion_spearman',
    'top_minus_random_relative_target_logit_reduction_auc',
    'top_beats_random_target_logit',
]
aggregate = (
    primary.groupby(['model', 'class_name'], as_index=False)[primary_metrics]
    .agg(['mean', 'median', 'std'])
)
aggregate.to_csv(STATISTICS_DIR / 'external_primary_faithfulness_by_class.csv')

overall = (
    primary.groupby(['model'], as_index=False)[primary_metrics]
    .agg(['mean', 'median', 'std'])
)
overall.to_csv(STATISTICS_DIR / 'external_primary_faithfulness_overall.csv')

all_image_aggregate = (
    all_image.groupby(['model', 'class_name'], as_index=False)[primary_metrics]
    .agg(['mean', 'median', 'std'])
)
all_image_aggregate.to_csv(
    STATISTICS_DIR / 'external_all_image_true_class_faithfulness_by_class.csv'
)
all_image_overall = (
    all_image.groupby(['model'], as_index=False)[primary_metrics]
    .agg(['mean', 'median', 'std'])
)
all_image_overall.to_csv(
    STATISTICS_DIR / 'external_all_image_true_class_faithfulness_overall.csv'
)

joint_tests, joint_paired_values = paired_external_inference(
    metrics,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    analysis_family='primary_jointly_correct_true_class',
)
all_image_tests, all_image_paired_values = paired_external_inference(
    metrics,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    analysis_family='all_image_true_class_sensitivity',
)
tests = pd.concat((joint_tests, all_image_tests), ignore_index=True)
paired_values = pd.concat(
    (joint_paired_values, all_image_paired_values), ignore_index=True
)
tests.to_csv(STATISTICS_DIR / 'external_paired_model_tests.csv', index=False)
paired_values.to_csv(STATISTICS_DIR / 'external_paired_model_values.csv', index=False)

display(overall.round(4))
display(all_image_overall.round(4))
display(
    tests[
        [
            'analysis_family', 'metric', 'model_a', 'model_b',
            'paired_images', 'mean_difference',
            'bootstrap_ci_low', 'bootstrap_ci_high',
            'bootstrap_inference_level', 'wilcoxon_p_holm',
        ]
    ].round(5)
)


## Completion record

Notebook 14 will turn these outputs into confusion matrices, class-specific
faithfulness figures, deletion curves, representative maps, and the final
Kather-versus-CRC generalization statement. Do not infer pathology-level or
biological causality from highlighted regions.


In [ ]:
metadata = {
    'study': 'Kather-to-CRC common-seven external attribution',
    'models': list(MODEL_ORDER),
    'seeds': list(SEEDS),
    'cohort_images': int(cohort['cohort_id'].nunique()),
    'cohort_selected_using_predictions': False,
    'evaluation_grid': 14,
    'perturbation': 'normalized_zero',
    'random_deletion_repeats': 5,
    'primary_analysis': 'jointly correct, common true class',
    'class_complete_complement': 'all-image common true-class sensitivity',
    'analysis_amendment_before_attribution': (
        'jointly correct subset lacks lymphocytes; report both families'
    ),
    'inference_level': readiness['inference_level'],
    'patient_metadata_available': readiness['patient_metadata_available'],
    'raw_cross_model_logit_comparison_primary': False,
    'biological_causality_claim_supported': False,
}
(OUTPUT_DIR / 'faithfulness_run_metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)
display(pd.Series(metadata, name='value').to_frame())
